# Gemini 3 Batch Transcription Pipeline

This Colab orchestrates the transcription of audio segments using Google's **Gemini 3** model.

**Runtime:** requires the repo on disk — run in a Jupyter kernel whose CWD is `model/colabs/` so the sibling `common/` package imports (the `model/notebook_docker/` compose service provides this). The stock *Open in Colab* badge was removed; this notebook is no longer one-click Colab.

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically checks for existing transcripts and skips already processed files to avoid duplicate work and optimize API costs (Delta processing).
2.  **Organization**: Results are written directly to GCS.

In [ ]:
!pip install loguru
!pip install "google-genai>=2.3,<3"

In [ ]:
import json
import re
import sys
import time

from google import genai
from google.cloud import storage
from google.colab import auth
from loguru import logger

if "." not in sys.path:
    sys.path.insert(0, ".")

from common.gemini.prompts import GEMINI_TRANSCRIBE_SYSTEM_PROMPT
from common.gemini.prompts import GEMINI_TRANSCRIBE_USER_PROMPT
from common.gemini.vertex import GEMINI_GENERATION_CONFIG, GEMINI_SAFETY_SETTINGS
from common.gemini.vertex import build_request, submit_batch_inference

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent
!gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
    --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

In [ ]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3.1-flash-lite-preview"  # @param ["gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}

GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = "one_hour_pilot"  # @param {type:"string"}
GCS_INPUT_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
EXPERIMENT_NAME = ""  # @param {type:"string"}
# @markdown Enable if modifications (e.g. silence padding) were applied during segmentation:
AUDIO_PREPROCESSING = True  # @param {type:"boolean"}

# Handle Preprocessing suffix
if not AUDIO_PREPROCESSING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"

# Validation: Ensure required fields are filled to avoid IndexError in downstream GCS calls
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in the form above."
assert GCS_BUCKET, "GCS_BUCKET must be provided in the form above."
assert PROJECT_NAME, "PROJECT_NAME must be provided."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided in the form above."

# Pipeline Control
OVERWRITE_EXISTING = True  # @param {type:"boolean"}

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
GCP_LOCATION = "global"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

BATCH_INPUT_URI = (
    f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/vertex_batch_input.jsonl"
)
BATCH_OUTPUT_ROOT = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/batch_results/"

# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"


logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [ ]:
# @title Experiment here — override canonical defaults for A/B testing
# Edit these to try prompt / inference-config variants in this session.
# Promote a winner by editing common/gemini/prompts.py or common/gemini/vertex.py and committing.

SYSTEM_PROMPT = (
    GEMINI_TRANSCRIBE_SYSTEM_PROMPT  # <- edit to A/B a prompt version
)
USER_PROMPT = GEMINI_TRANSCRIBE_USER_PROMPT
GENERATION_CONFIG = {
    **GEMINI_GENERATION_CONFIG
}  # <- edit temperature / max_output_tokens
SAFETY_SETTINGS = GEMINI_SAFETY_SETTINGS

In [ ]:
# @title Helper functions


def get_failed_uris(results_uri: str, bucket_name: str) -> list[str]:
    """Identifies URIs that resulted in errors in the final output file."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    path = results_uri.replace(f"gs://{bucket_name}/", "")
    blob = storage_client.bucket(bucket_name).blob(path)
    if not blob.exists():
        return []
    lines = blob.download_as_text().strip().split("\n")
    failed = []
    for line in lines:
        if not line.strip():
            continue
        data = json.loads(line)
        if data.get("status"):
            parts = data["request"]["contents"][0]["parts"]
            uri = next(
                (
                    p.get("file_data", {}).get("file_uri")
                    or p.get("fileData", {}).get("fileUri")
                    for p in parts
                    if "file_uri" in str(p) or "fileUri" in str(p)
                ),
                "unknown",
            )
            failed.append(uri)
    return failed


def create_retry_manifest(
    failed_uris: list[str], original_manifest_uri: str, retry_manifest_uri: str
) -> None:
    """Filters original manifest and wraps in the correct 'request' structure for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket_name = original_manifest_uri.replace("gs://", "").split("/")[0]
    path = "/".join(original_manifest_uri.replace("gs://", "").split("/")[1:])
    content = (
        storage_client.bucket(bucket_name)
        .blob(path)
        .download_as_text()
        .strip()
        .split("\n")
    )

    retry_entries = []
    for line in content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in failed_uris:
            batch_entry = build_request(
                entry["audio_filepath"],
                system_prompt=SYSTEM_PROMPT,
                user_prompt=USER_PROMPT,
                generation_config=GENERATION_CONFIG,
                safety_settings=SAFETY_SETTINGS,
            )
            retry_entries.append(json.dumps(batch_entry))

    if retry_entries:
        out_bucket = retry_manifest_uri.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            retry_manifest_uri.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            "\n".join(retry_entries)
        )
        logger.info(
            f"Uploaded retry manifest with {len(retry_entries)} entries."
        )


def prepare_batch_manifest(
    input_manifest_uri: str,
    output_batch_manifest_uri: str,
    *,
    overwrite: bool = False,
    limit: int | None = None,
) -> str | None:
    """Prepares the JSONL manifest with correct structural requirements for Vertex Batch."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = input_manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(input_manifest_uri.replace("gs://", "").split("/")[1:])

    try:
        manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
        if not manifest_blob.exists():
            logger.error(
                f"Manifest not found at {input_manifest_uri}. Check AUDIO_PREPROCESSING settings."
            )
            return None
        manifest_content = manifest_blob.download_as_text().strip().split("\n")
    except Exception as e:
        logger.error(f"Error reading manifest: {e}")
        return None

    processed_uris = set()
    if not overwrite:
        r_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        r_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        results_blob = storage_client.bucket(r_bucket).blob(r_path)
        if results_blob.exists():
            for line in results_blob.download_as_text().strip().split("\n"):
                if line.strip():
                    data = json.loads(line)
                    if not data.get("status"):
                        contents = data["request"].get("contents", [])
                        parts = (
                            contents[0].get("parts", [])
                            if isinstance(contents, list) and contents
                            else contents.get("parts", [])
                        )
                        uri = next(
                            (
                                p.get("file_data", {}).get("file_uri")
                                or p.get("fileData", {}).get("fileUri")
                                for p in parts
                                if "file_uri" in str(p) or "fileUri" in str(p)
                            ),
                            None,
                        )
                        if uri:
                            processed_uris.add(uri)

    batch_entries = []
    for line in manifest_content:
        if not line.strip():
            continue
        entry = json.loads(line)
        if entry["audio_filepath"] in processed_uris:
            continue

        batch_entry = build_request(
            entry["audio_filepath"],
            system_prompt=SYSTEM_PROMPT,
            user_prompt=USER_PROMPT,
            generation_config=GENERATION_CONFIG,
            safety_settings=SAFETY_SETTINGS,
        )
        batch_entries.append(json.dumps(batch_entry))
        if limit and len(batch_entries) >= limit:
            logger.info(f"Test limit of {limit} reached.")
            break

    if not batch_entries:
        return None

    out_bucket = output_batch_manifest_uri.replace("gs://", "").split("/")[0]
    out_path = "/".join(
        output_batch_manifest_uri.replace("gs://", "").split("/")[1:]
    )
    storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
        "\n".join(batch_entries)
    )
    logger.info(
        f"Prepared {len(batch_entries)} segments for processing at {output_batch_manifest_uri}"
    )
    return output_batch_manifest_uri


def run_automated_retry_pipeline() -> None:
    """Orchestrates checking for failures, creating a retry manifest, and merging results."""
    logger.info("Starting automated error check...")
    failed_uris = get_failed_uris(CONSISTENT_OUTPUT_URI, GCS_BUCKET)
    if not failed_uris:
        logger.info("No failed segments detected. Pipeline complete.")
        return

    logger.info(
        f"Detected {len(failed_uris)} failures. Creating retry manifest... "
    )
    RETRY_MANIFEST = (
        f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/automated_retry_manifest.jsonl"
    )
    create_retry_manifest(failed_uris, MANIFEST_URI, RETRY_MANIFEST)

    try:
        submit_batch_inference(
            input_uri=RETRY_MANIFEST,
            output_uri=BATCH_OUTPUT_ROOT,
            model=MODEL_ID,
            project=GCP_PROJECT_ID,
            location=GCP_LOCATION,
        )
        consolidate_all_successes(
            GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
        )
    except RuntimeError as e:
        logger.error(f"Retry job failed: {e}")
    validate_transcription_results(
        MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
    )


def consolidate_all_successes(
    bucket_name: str, output_base: str, target_uri: str
) -> None:
    """Scans all batch result folders and builds a unique map of successful transcriptions."""
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    prefix = f"{output_base}/batch_results/"
    blobs = bucket.list_blobs(prefix=prefix)

    success_map = {}
    for blob in blobs:
        if "predictions.jsonl" not in blob.name:
            continue

        lines = blob.download_as_text().strip().split("\n")
        for line in lines:
            if not line.strip():
                continue
            data = json.loads(line)
            if not data.get("status"):
                parts = data["request"]["contents"][0]["parts"]
                uri = next(
                    (
                        p.get("file_data", {}).get("file_uri")
                        or p.get("fileData", {}).get("fileUri")
                        for p in parts
                        if "file_uri" in str(p) or "fileUri" in str(p)
                    ),
                    "unknown",
                )
                if uri not in success_map:
                    success_map[uri] = line

    target_path = target_uri.replace(f"gs://{bucket_name}/", "")
    bucket.blob(target_path).upload_from_string("\n".join(success_map.values()))
    logger.info(
        f"Consolidation complete. Total unique successful segments: {len(success_map)}"
    )


def validate_transcription_results(
    manifest_uri: str, results_uri: str, bucket_name: str
) -> None:
    storage_client = storage.Client(project=GCP_PROJECT_ID)
    m_bucket = manifest_uri.replace("gs://", "").split("/")[0]
    m_path = "/".join(manifest_uri.replace("gs://", "").split("/")[1:])

    m_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not m_blob.exists():
        logger.warning(
            f"Validation skipped: Manifest {manifest_uri} does not exist."
        )
        return

    expected = len(
        [
            line
            for line in m_blob.download_as_text().strip().split("\n")
            if line.strip()
        ]
    )
    r_path = results_uri.replace(f"gs://{bucket_name}/", "")
    results_blob = storage_client.bucket(bucket_name).blob(r_path)
    if results_blob.exists():
        found = len(
            [
                line
                for line in results_blob.download_as_text().strip().split("\n")
                if line.strip()
            ]
        )
        if found == expected:
            logger.info(
                f"Pipeline Validation: SUCCESS. Expected {expected}, Found {found}."
            )
        else:
            logger.error(
                f"Pipeline Validation: FAILURE. Expected {expected}, Found {found}."
            )

In [ ]:
# @title Main Job Submission
TEST_RUN = False  # @param {type:"boolean"}
TEST_LIMIT = 2  # @param {type:"integer"}

actual_batch_input = prepare_batch_manifest(
    input_manifest_uri=MANIFEST_URI,
    output_batch_manifest_uri=BATCH_INPUT_URI,
    overwrite=OVERWRITE_EXISTING,
    limit=TEST_LIMIT if TEST_RUN else None,
)

if actual_batch_input:
    logger.info("Submitting main batch job...")
    try:
        submit_batch_inference(
            input_uri=actual_batch_input,
            output_uri=BATCH_OUTPUT_ROOT,
            model=MODEL_ID,
            project=GCP_PROJECT_ID,
            location=GCP_LOCATION,
        )
        logger.info("Main job completed. Consolidating results...")
        consolidate_all_successes(
            GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
        )
        validate_transcription_results(
            MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
        )
    except RuntimeError as e:
        logger.error(f"Main job failed: {e}. Skipping consolidation.")
else:
    logger.info(
        "Skipping main job submission (no new segments to process or manifest missing). Attempting consolidation of previous results..."
    )
    consolidate_all_successes(
        GCS_BUCKET, GCS_OUTPUT_BASE, CONSISTENT_OUTPUT_URI
    )
    validate_transcription_results(
        MANIFEST_URI, CONSISTENT_OUTPUT_URI, GCS_BUCKET
    )

In [ ]:
# @title Automated Retry & Merge Orchestrator
logger.info("Executing automated retry and merge pipeline...")
run_automated_retry_pipeline()

In [ ]:
# @title Write run metadata
import datetime as _datetime

_run_metadata = {
    "prompt": SYSTEM_PROMPT,
    "model_id": MODEL_ID,
    "audio_preprocessing": AUDIO_PREPROCESSING,
    "generation_config": GENERATION_CONFIG,
    "safety_settings": SAFETY_SETTINGS,
    "project_name": PROJECT_NAME,
    "experiment_name": EXPERIMENT_NAME,
    "input_manifest_uri": MANIFEST_URI,
    "predictions_uri": CONSISTENT_OUTPUT_URI,
    "run_timestamp_utc": _datetime.datetime.now(
        _datetime.timezone.utc
    ).strftime("%Y-%m-%dT%H:%M:%S.%fZ"),
}

_meta_path = f"{GCS_OUTPUT_BASE}/run_metadata.json"
storage.Client(project=GCP_PROJECT_ID).bucket(GCS_BUCKET).blob(
    _meta_path
).upload_from_string(
    json.dumps(_run_metadata, indent=2),
    content_type="application/json",
)
logger.info(f"Run metadata written to gs://{GCS_BUCKET}/{_meta_path}")